# 07 — Calibrate 4 panels with `midas_calibrate_v2`, then run multi-detector FF-HEDM

**Scenario.** A user has:

1. **4 calibrant images** (one per detector panel) of CeO₂ at the alignment energy.
2. **A few-grain HEDM dataset** acquired in the same geometry as the calibrant — **stored as raw per-frame TIFFs**, e.g. 1440 TIFFs per panel (one per ω step). No pre-packed `.MIDAS.zip`s.
3. **A pre-survey** of each panel's azimuthal mount angle `tx` around the beam — typically known to **~1°** from the encoder/metrology, but not better. Powder calibration on one panel can't tighten `tx` below the survey value, because `tx` mostly rotates the powder pattern in-plane and is degenerate with `ty/tz` on a single panel.

**Why a one-shot calibrate-then-run is suboptimal.** At ±1° of `tx` error the pipeline *will* index — production tolerances are wide enough to absorb the resulting spot drift — but you'll see asymmetric per-panel spot counts, modestly elevated strain residuals, and a few percent fewer grains than you could be getting. The fix is a quick **two-pass bootstrap**: run with slightly relaxed tolerances, harvest the indexed grains, joint-refine per-panel `tx` (and pick up global `Wedge` for free), then re-run at production tolerances.

**Workflow.**

1. (Optional) Peek at one raw TIFF per panel with **fabIO** to confirm folder/panel mapping.
2. Per-panel powder calibration (`midas_calibrate_v2.autocalibrate_pv`) — fixes `Lsd / BC / ty / tz / distortion`. `tx` stays at the survey value.
3. Pack each panel's TIFF stack into a `.MIDAS.zip`.
4. Write `detectors.json` (with the survey `tx`) + `Parameters.txt` with **mildly relaxed** tolerances (just enough to absorb ±1° on `tx`).
5. **Pass 1**: run the pipeline. Most grains will index; per-panel spot counts may be uneven.
6. **Joint refinement** from the pass-1 grains: refine per-panel `tx` (and optionally global `Wedge`) using `midas_joint_ff_calibrate.AlternatingDriver`. This pushes `tx` from ~1° down to arcminute precision — something powder alone cannot do.
7. Write a new `detectors.json` with the refined `tx` and a new `Parameters.txt` with **production-tight** tolerances.
8. **Pass 2**: re-run the pipeline. Expect symmetric per-panel spot counts, lower strain residuals, and slightly more grains.

> **fabIO tip.** [fabIO](https://github.com/silx-kit/fabio) (`pip install fabio`) is the easiest way to inspect raw detector files in any common format — TIFF, GE5, EDF, CBF, MAR, HDF5, etc. `fabio.open(path).data` returns a NumPy array; `fabio.open(path).header` gives metadata; `reader.nframes` walks multi-frame containers. The pipeline's `zip_convert` stage uses its own readers, but fabIO is invaluable for QA.

Substitute your own paths in the `INPUTS` cell and run.

## 0 — Inputs

In [ ]:
from pathlib import Path

WORK = Path('/tmp/midas_4det_demo')
WORK.mkdir(parents=True, exist_ok=True)

# --- Calibrant: one .tif per panel ----------------------------------------
CALIBRANT_IMAGES = {
    1: Path('/data/ceo2/panel1.tif'),
    2: Path('/data/ceo2/panel2.tif'),
    3: Path('/data/ceo2/panel3.tif'),
    4: Path('/data/ceo2/panel4.tif'),
}
DARK_IMAGES = {                       # optional; set to None to skip
    1: Path('/data/ceo2/dark1.tif'),
    2: Path('/data/ceo2/dark2.tif'),
    3: Path('/data/ceo2/dark3.tif'),
    4: Path('/data/ceo2/dark4.tif'),
}

# --- HEDM data: raw per-frame TIFFs per panel -----------------------------
# Each panel has its own RawFolder + FileStem; frame index runs StartNr..EndNr,
# zero-padded to Padding digits with extension Ext.
#   e.g.  /data/sample/panel1/scan1_000001.tif … scan1_001440.tif
HEDM_TIFFS = {
    1: dict(raw_folder='/data/sample/panel1', file_stem='scan1', start_nr=1, end_nr=1440, padding=6, ext='.tif'),
    2: dict(raw_folder='/data/sample/panel2', file_stem='scan1', start_nr=1, end_nr=1440, padding=6, ext='.tif'),
    3: dict(raw_folder='/data/sample/panel3', file_stem='scan1', start_nr=1, end_nr=1440, padding=6, ext='.tif'),
    4: dict(raw_folder='/data/sample/panel4', file_stem='scan1', start_nr=1, end_nr=1440, padding=6, ext='.tif'),
}
HEDM_DARKS = {                        # optional; one .tif per panel for dark subtraction
    1: '/data/sample/panel1/dark.tif',
    2: '/data/sample/panel2/dark.tif',
    3: '/data/sample/panel3/dark.tif',
    4: '/data/sample/panel4/dark.tif',
}
OMEGA_START =   -180.0     # degrees
OMEGA_STEP  =      0.25    # degrees per frame

# --- Rough initial geometry (shared across panels except tx) --------------
INITIAL = dict(
    Lsd          = 1_000_000.0,   # μm, sample-to-detector
    BC           = (1024.0, 1024.0),   # (y, z) pixels — assume centred for now
    px           = 200.0,         # μm, pixel size
    Wavelength   = 0.1729,        # Å, ~71.7 keV
    NrPixelsY    = 2048,
    NrPixelsZ    = 2048,
    RhoD         = 200_000.0,     # μm, max radius from beam centre
    Material     = 'CeO2',
    LatticeConstant = (5.4116, 5.4116, 5.4116, 90.0, 90.0, 90.0),
    SpaceGroup   = 225,
    # tilts (degrees) — tx is the azimuthal mount angle of each panel
    ty_init      = 0.0,
    tz_init      = 0.0,
)

# Panel-specific tx from the pre-survey — known to ~1° from the mount encoder/metrology.
# Joint refinement in step 9 will push these to arcminute precision.
TX_GUESS = {1: 0.0, 2: 90.0, 3: 180.0, 4: 270.0}

# Sample of the actual material we'll be indexing (often different from calibrant).
SAMPLE = dict(
    Material        = 'Cu',
    LatticeConstant = (3.615, 3.615, 3.615, 90.0, 90.0, 90.0),
    SpaceGroup      = 225,
)

print(f'work dir: {WORK}')

## 1 — Peek at the raw frames with fabIO (optional)

Quick sanity check before committing to a full run. Open the **first** TIFF of each panel plus one calibrant frame, and confirm shape, intensity range, and that you didn't mix up which folder is which detector. `fabio.open(...).data` is a plain NumPy array — same handling whether the underlying file is `.tif`, `.ge5`, `.cbf`, `.edf`, `.mar`, or `.h5`.

In [ ]:
import fabio
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, det_id in enumerate(sorted(HEDM_TIFFS)):
    cfg   = HEDM_TIFFS[det_id]
    first = Path(cfg['raw_folder']) / f"{cfg['file_stem']}_{cfg['start_nr']:0{cfg['padding']}d}{cfg['ext']}"
    img   = fabio.open(str(first)).data.astype(np.float32)
    axes[0, col].imshow(img, cmap='gray', origin='lower',
                        vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
    axes[0, col].set_title(f'HEDM det {det_id}\n{first.name}\n{img.shape}')
    axes[0, col].axis('off')

    calib = CALIBRANT_IMAGES[det_id]
    cimg  = fabio.open(str(calib)).data.astype(np.float32)
    axes[1, col].imshow(cimg, cmap='gray', origin='lower',
                        vmin=np.percentile(cimg, 1), vmax=np.percentile(cimg, 99.5))
    axes[1, col].set_title(f'CeO₂ det {det_id}\n{calib.name}')
    axes[1, col].axis('off')
plt.tight_layout(); plt.show()

# Optional: walk the multi-frame container of one panel to confirm frame count.
#   reader = fabio.open(str(first))
#   print('nframes:', reader.nframes)
#   print('header :', reader.header)

## 2 — Build one seed `paramstest.txt` per panel

`midas_calibrate_v2` reads a v1 `CalibrationParams` object as its seed. The fields below are the minimum a calibration run needs; everything else inherits the dataclass default.

In [ ]:
from midas_calibrate.params import CalibrationParams as V1Params

def make_seed_params(det_id: int) -> V1Params:
    p = V1Params()
    p.Lsd        = INITIAL['Lsd']
    p.BC         = list(INITIAL['BC'])
    p.px         = INITIAL['px']
    p.pxY        = INITIAL['px']
    p.pxZ        = INITIAL['px']
    p.Wavelength = INITIAL['Wavelength']
    p.NrPixelsY  = INITIAL['NrPixelsY']
    p.NrPixelsZ  = INITIAL['NrPixelsZ']
    p.RhoD       = INITIAL['RhoD']
    p.MaxRingRad = INITIAL['RhoD'] / INITIAL['px']
    p.Material   = INITIAL['Material']
    p.LatticeConstant = list(INITIAL['LatticeConstant'])
    p.SpaceGroup = INITIAL['SpaceGroup']
    p.tx         = TX_GUESS[det_id]
    p.ty         = INITIAL['ty_init']
    p.tz         = INITIAL['tz_init']
    p.RBinSize   = 0.25
    p.EtaBinSize = 5.0
    p.Width      = 1000.0
    return p

seed_params = {d: make_seed_params(d) for d in CALIBRANT_IMAGES}
for d, p in seed_params.items():
    print(f'det {d}: tx={p.tx:>6.1f}°  Lsd={p.Lsd:.0f}μm  BC={p.BC}')

## 3 — Powder-calibrate each panel with `autocalibrate_pv`

Production single-image pipeline: cake + pseudo-Voigt peak fits in the E-step, Levenberg–Marquardt minimisation of the pseudo-strain residual in the M-step. Defaults refine `Lsd`, `BC`, `ty`, `tz`, and the leading distortion harmonics.

**`tx` is intentionally left frozen here** — powder rings cannot constrain `tx` independently of `ty/tz` on a single panel that sits in the centre of the ring system, and even when they can the SNR on `tx` is poor. We'll refine `tx` jointly across all four panels in step 9 using the indexed grains from pass 1.

Each call processes one panel; we loop and collect results. (Loader is fabIO-based, so swap to GE5/CBF/EDF transparently.)

In [ ]:
from midas_calibrate_v2.compat.from_v1 import spec_from_v1
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv

def load_image(path: Path, dark_path: Path | None) -> np.ndarray:
    """fabIO-based reader; MIDAS convention is Y horizontal, Z vertical → vertical-flip."""
    img = fabio.open(str(path)).data.astype(np.float64)[::-1, :].copy()
    if dark_path is not None:
        dark = fabio.open(str(dark_path)).data.astype(np.float64)[::-1, :].copy()
        img = img - dark
    return img

results = {}
for det_id, image_path in CALIBRANT_IMAGES.items():
    print(f'\n── det {det_id}: calibrating from {image_path.name} ──')
    v1   = seed_params[det_id]
    img  = load_image(image_path, DARK_IMAGES.get(det_id))
    spec = spec_from_v1(v1)   # default refined set: Lsd, BC_y, BC_z, ty, tz, distortion
    # NOTE: tx stays frozen — refined jointly in step 9 instead.

    res = autocalibrate_pv(
        v1, img, spec=spec,
        n_iter=4,
        half_window_px=8.0,
        snr_min=8.0,
        trim_mode='stratified_multfactor',
        trim_residual_pct=5.0,
        reuse_fits=True,
        lm_max_iter=300,
        verbose=False,
    )
    results[det_id] = res

    u = res.unpacked
    print(f'  Lsd={float(u["Lsd"]):.1f}μm  BC=({float(u["BC_y"]):.2f}, {float(u["BC_z"]):.2f})  '
          f'ty={float(u["ty"]):.3f}°  tz={float(u["tz"]):.3f}°')

### Optional — write a v1-style `paramstest_refined.txt` per panel

In [ ]:
from midas_calibrate_v2.compat.to_v1 import write_v1_paramstest

refined_params_files = {}
for det_id, res in results.items():
    out = WORK / f'paramstest_refined_det{det_id}.txt'
    write_v1_paramstest(res.unpacked, seed_params[det_id], out)
    refined_params_files[det_id] = out
    print(f'det {det_id}: wrote {out}')

## 4 — Pack each panel's TIFF stack into a `.MIDAS.zip`

The pipeline's per-panel readers expect a single zarr bundle per detector. We invoke `utils/ffGenerateZipRefactor.py` once per panel with a small per-panel params file telling it where the TIFFs live (`RawFolder`, `FileStem`, `Padding`, `Ext`, `StartFileNrFirstLayer`, `NrFilesPerSweep`) and the ω metadata to bake in.

> **fabIO sidenote.** If your raw data is **not** TIFF — e.g. GE5 from APS or CBF from Diamond — change `Ext` and `ffGenerateZipRefactor.py` will pick the right reader. fabIO can verify intensity scaling and headers before you commit.

In [ ]:
import subprocess, sys
import midas_ff_pipeline

MIDAS_ROOT = Path(midas_ff_pipeline.__file__).resolve().parents[3]
ZIP_SCRIPT = MIDAS_ROOT / 'utils' / 'ffGenerateZipRefactor.py'
assert ZIP_SCRIPT.exists(), f'cannot find {ZIP_SCRIPT}'

PACK_DIR = WORK / 'zarrs';  PACK_DIR.mkdir(exist_ok=True)
packed_zips = {}

for det_id, cfg in HEDM_TIFFS.items():
    panel_params = WORK / f'pack_det{det_id}.txt'
    overlay = f'''
RawFolder              {cfg['raw_folder']}
FileStem               {cfg['file_stem']}
Padding                {cfg['padding']}
Ext                    {cfg['ext']}
StartFileNrFirstLayer  {cfg['start_nr']}
NrFilesPerSweep        {cfg['end_nr'] - cfg['start_nr'] + 1}
OmegaStart             {OMEGA_START}
OmegaStep              {OMEGA_STEP}
'''
    if det_id in HEDM_DARKS:
        overlay += f'darkFN                 {HEDM_DARKS[det_id]}\n'
    panel_params.write_text(refined_params_files[det_id].read_text() + overlay)

    panel_out = PACK_DIR / f'det{det_id}'
    panel_out.mkdir(exist_ok=True)
    print(f'\n── det {det_id}: packing TIFFs → .MIDAS.zip ──')
    subprocess.run(
        [sys.executable, str(ZIP_SCRIPT),
         '-resultFolder', str(panel_out),
         '-paramFN',      str(panel_params),
         '-LayerNr',      '1'],
        check=True,
    )
    zips = sorted(panel_out.rglob('*.MIDAS.zip'), key=lambda p: p.stat().st_mtime)
    if not zips:
        raise FileNotFoundError(f'no .MIDAS.zip produced under {panel_out}')
    packed_zips[det_id] = zips[-1]
    print(f'  wrote {packed_zips[det_id]}')

## 5 — Assemble `detectors.json` (pass-1, with rough `tx`)

[`DetectorConfig`](../midas_ff_pipeline/detector.py): μm for `lsd`, pixels for `y_bc/z_bc`, degrees for `tx/ty/tz`. `p_distortion` is 11 radial harmonics. `tx` is the **rough guess** here — we'll overwrite it in step 9.

In [ ]:
from midas_ff_pipeline import DetectorConfig

def to_detector_config(det_id: int, res, zarr_path: Path, tx_override: float | None = None) -> DetectorConfig:
    u = res.unpacked
    p_dist = [float(u[f'p{i}']) if f'p{i}' in u else 0.0 for i in range(11)]
    return DetectorConfig(
        det_id   = det_id,
        zarr_path= str(zarr_path),
        lsd      = float(u['Lsd']),
        y_bc     = float(u['BC_y']),
        z_bc     = float(u['BC_z']),
        tx       = float(tx_override) if tx_override is not None else float(TX_GUESS[det_id]),
        ty       = float(u['ty']),
        tz       = float(u['tz']),
        p_distortion = p_dist,
    )

detectors_p1 = [to_detector_config(d, results[d], packed_zips[d]) for d in sorted(CALIBRANT_IMAGES)]
detectors_json_p1 = WORK / 'detectors_pass1.json'
DetectorConfig.dump_many(detectors_p1, detectors_json_p1)
print(f'wrote {detectors_json_p1}')
for d in detectors_p1:
    print(f'  det {d.det_id}: Lsd={d.lsd:.0f}μm  tx={d.tx:.2f}° (guess)  ty={d.ty:.3f}°  tz={d.tz:.3f}°')

## 6 — `Parameters.txt` for **pass 1** (loosened for both eta-bias and position fuzz)

`tx` error has **two** effects on residuals, and both need slack in pass-1:

1. **Per-spot eta residuals get a uniform bias on the offending panel**, of order $\Delta_\text{tx} \times R_\text{ring}$. For a 1° tx error on the outermost ring at 200 mm, that's ~3500 μm of azimuthal displacement per spot. `MarginEta` must comfortably exceed this — otherwise spots get rejected before reaching the indexer.
2. **Grain-position triangulation disagreement.** Back-projecting a grain's lab `(X, Y, Z)` from one panel's biased spots vs another panel's unbiased spots produces inconsistent answers — the back-projections disagree on where the grain is. The exact magnitude depends on panel layout, ring coverage, and how many spots per grain come from the biased panel; we don't have a clean closed-form for it. Be conservative and widen `BoxSize` enough that the search will still contain the grain even when one panel's contribution drags the position estimate off.

Pass-2 reverts both back to production.

In [ ]:
params_template = refined_params_files[1]
params_pass1 = WORK / 'Parameters_pass1.txt'

# Pass-1 budget at ±1° on tx:
#  - MarginEta sized to comfortably exceed Δtx × R_outer_ring (~3500 μm at 1° × 200 mm)
#  - BoxSize / StepSizePos: generous slack — grain-position triangulation disagreement
#    has no clean closed form, so be conservative
#  - tolTilts/tolBC/tolLsd: loosened so the geometry refinement can move during pass-1
loose_extra = f'''
# --- sample (overrides calibrant material) --------------------------------
Material         {SAMPLE['Material']}
LatticeConstant  {' '.join(map(str, SAMPLE['LatticeConstant']))}
SpaceGroup       {SAMPLE['SpaceGroup']}

# --- omega + box (geometry of the scan) -----------------------------------
OmegaStart      {OMEGA_START}
OmegaStep       {OMEGA_STEP}
OmegaRange      {OMEGA_START}  {OMEGA_START + (1440 * OMEGA_STEP)}

# Position search box — conservative slack for grain-position disagreement
# between panels with mis-set tx.  If pass-1 still misses grains, widen further.
BoxSize         -5000000 5000000 -5000000 5000000        # μm; production maybe ±300 to ±1000

# --- spot-matching margins (eta-side gets the BIG bump for tx) ------------
MarginOme        0.7        # production 0.5  → just a small bump
MarginEta       5000        # μm, production 500  → must cover Δtx × R_outer ≈ 3500 μm
MarginRadial     500        # μm, production 400  → tx barely shifts radial
MarginRadius     600        # production 500

# --- position-side knobs (generous slack) ---------------------------------
MinNrSpots       40         # production 60 — accept partially-found grains
StepSizePos      400        # μm, production 100 — coarser grain-position grid
StepSizeOrient   0.3        # degrees, production 0.2 — barely loose

# --- geometry-refinement tolerances (loose; expect drift) -----------------
DiscModel        0
tolTilts         3.0        # degrees, production 2.0
tolBC           20.0        # production 10
tolLsd        1000.0        # μm, production 500
RingsToUse       1 2 3 4 5 6 7
'''

params_pass1.write_text(params_template.read_text() + loose_extra)
print(f'wrote {params_pass1}')

## 7 — Run the pipeline (pass 1)

[`Pipeline`](../midas_ff_pipeline/pipeline.py) executes:

1. `zip_convert` (per panel) — **skipped**: zarrs already exist.
2. `peakfit` → `merge_overlaps` → `calc_radius` → `transforms` (per panel)
3. `cross_det_merge` → `global_powder` → `binning` → `indexing` → `refinement` → `process_grains` (global)

At ±1° on `tx`, most grains should index successfully on pass-1 — the relaxed `BoxSize` and `StepSizePos` absorb the position-triangulation fuzz, and spot margins stay near production. The pass-1 grain list and matched-spot bookkeeping is what step 9 will refine `tx` and `Wedge` against.

In [ ]:
from midas_ff_pipeline import Pipeline, PipelineConfig

pipe_p1 = Pipeline(
    config=PipelineConfig(
        result_dir     = str(WORK / 'run_pass1'),
        params_file    = str(params_pass1),
        detectors_json = str(detectors_json_p1),
        n_cpus         = 8,
        device         = 'cpu',
        dtype          = 'float64',
    ),
    detectors=detectors_p1,
)
pipe_p1.run()
result_p1 = pipe_p1.layer_result
grains_p1 = result_p1.grains_df()
print(f'\npass-1: {len(grains_p1)} grains, '
      f'{result_p1.cross_det_merge.n_total_spots:,} merged spots, '
      f'{result_p1.total_duration_s():.1f}s')
grains_p1.head()

## 8 — Inspect pass-1: per-panel eta residuals are the real diagnostic

A panel with `tx` off by $\Delta_\text{tx}$ has all its matched spots biased by ~$\Delta_\text{tx} \times R_\text{ring}$ in the azimuthal direction. So the **per-panel mean eta residual** (after pass-1 indexing has matched observed to predicted spots) is the strong signal: it's directly proportional to that panel's `tx` error, and — unlike spot counts, which can legitimately be asymmetric for non-symmetric mounts — the *mean* eta residual is geometry-bias-free. A panel with the correct `tx` should have mean eta residual ≈ 0; a panel with `tx` off by 0.5° will have a few-thousand-μm bias at the outer ring.

Plot the per-panel eta-residual distribution and the per-grain `NrSpotsMatched` distribution — these two together tell you (a) which panel needs the biggest `tx` correction in step 9 and (b) whether pass-1 went smoothly enough that the refinement will be well-conditioned.

In [ ]:
import pandas as pd

# ── (a) per-panel mean eta residual — the strong tx diagnostic ────────────
# SpotMatrix.csv (written by ProcessGrains/refinement) carries per-spot
# observed and predicted positions plus the detector each spot came from.
# Column names vary by MIDAS version — adapt the keys below if necessary.
sm = pd.read_csv(pipe_p1.layer_dir / 'SpotMatrix.csv')

# eta (degrees) from lab-frame Y, Z (observed and predicted)
eta_obs  = np.degrees(np.arctan2(sm['ObsZ'],  sm['ObsY']))
eta_pred = np.degrees(np.arctan2(sm['PredZ'], sm['PredY']))
# tangential residual in μm at the matched radius (more intuitive than Δη in deg)
R        = np.hypot(sm['ObsY'], sm['ObsZ'])
eta_res  = (eta_obs - eta_pred)                    # degrees
tan_res  = np.deg2rad(eta_res) * R                 # μm tangential displacement

print('per-panel eta-residual summary (μm tangential):')
print(f"  {'det':>4} {'N':>8} {'mean':>10} {'std':>10}  ← mean ≈ Δtx × R")
for d in detectors_p1:
    sel = sm['DetectorID'].to_numpy() == d.det_id
    if sel.sum() == 0: continue
    print(f"  {d.det_id:>4} {sel.sum():>8d} {tan_res[sel].mean():>+10.1f} {tan_res[sel].std():>10.1f}")

# ── (b) histograms ────────────────────────────────────────────────────────
fig, (ax_eta, ax_n) = plt.subplots(1, 2, figsize=(14, 5))

for d in detectors_p1:
    sel = sm['DetectorID'].to_numpy() == d.det_id
    if sel.sum() == 0: continue
    ax_eta.hist(tan_res[sel], bins=60, alpha=0.5, label=f'det {d.det_id}', histtype='stepfilled')
ax_eta.axvline(0, color='k', lw=0.5)
ax_eta.set_xlabel('tangential residual (μm)  =  Δη · R')
ax_eta.set_ylabel('count')
ax_eta.set_title('Pass-1 per-panel eta residuals\n(mean ≠ 0 → tx bias on that panel)')
ax_eta.legend()

if 'NrSpotsMatched' in grains_p1.columns:
    ax_n.hist(grains_p1['NrSpotsMatched'], bins=30, edgecolor='k', alpha=0.7)
    ax_n.axvline(40, color='r', ls='--', label='MinNrSpots (pass 1)')
    ax_n.set_xlabel('NrSpotsMatched per grain'); ax_n.set_ylabel('count')
    ax_n.legend(); ax_n.set_title('Pass-1 grain quality')
plt.tight_layout(); plt.show()

## 9 — HEDM-only refinement of per-panel `tx` (and optional `Wedge`)

Powder is done — it gave us `Lsd / BC / ty / tz / distortion` per panel and we won't touch them again. From here on, **only the HEDM diffraction spots constrain geometry.** This is the workflow demonstrated in the differentiable-HEDM paper ([`midas_diffract/dev/paper/scripts/run_4det_refinement.py`](../../midas_diffract/dev/paper/scripts/run_4det_refinement.py)) and is the right tool for tightening `tx` from ~1° down to ~arcminute precision.

The forward model:

- [`HEDMGeometry`](../../midas_diffract/midas_diffract/forward.py) ([`forward.py:43`](../../midas_diffract/midas_diffract/forward.py#L43)) accepts per-detector `Lsd / y_BC / z_BC / tx / ty / tz` lists with `multi_mode="panel"`, plus a global `wedge`.
- [`HEDMForwardModel`](../../midas_diffract/midas_diffract/forward.py) ([`forward.py:190`](../../midas_diffract/midas_diffract/forward.py#L190)) stores those as differentiable parameters: `model.tilts` (D×3, the per-panel `tx, ty, tz`), `model._Lsd_delta_mm` (per-panel offset in mm), `model._y_BC`, `model._z_BC`, `model.wedge` (global, deg).
- Toggle which params are refined via `configure_freedom(model, free_setup=[...])` ([`run_4det_refinement.py:192`](../../midas_diffract/dev/paper/scripts/run_4det_refinement.py#L192)).

**Schedule.** Adam warmup → L-BFGS polish. The per-parameter gradient magnitudes for `tilts` vs grain Eulers are imbalanced by ~50× ([paper §4.2](../../midas_diffract/dev/paper/)), so the warmup uses different learning rates per param group; L-BFGS then handles the final quadratic basin cleanly.

**Tx-only mode.** If you trust the powder `ty/tz` and only want `tx` to move, mask the ty/tz columns of `model.tilts.grad` with a gradient hook. This is exactly what scenario A in the paper script does ([`run_4det_refinement.py:430`](../../midas_diffract/dev/paper/scripts/run_4det_refinement.py#L430)).

**Spot bookkeeping.** [`SpotIndex`](../../midas_diffract/dev/paper/scripts/run_4det_refinement.py#L222) maps the pass-1 spot CSV into `(grain_idx, k_idx, m_idx)` triples so the loss is a direct elementwise comparison against the forward-model output. In a production wrapper this is replaced by [`midas_fit_grain.spec_residual.hedm_spot_residual`](../../midas_fit_grain/midas_fit_grain/spec_residual.py) ([`spec_residual.py:112`](../../midas_fit_grain/midas_fit_grain/spec_residual.py#L112)), which supports per-detector `tilts_per_det / Lsd_per_det / BC_y_per_det / BC_z_per_det` overrides via `torch.func.functional_call` and returns a residual vector with autograd flow back to all of them.

In [ ]:
import torch
from midas_diffract import HEDMForwardModel, HEDMGeometry

# ── (1) Build per-panel HEDM geometry from pass-1 detectors ───────────────
# Stack the 4 panels into list-valued geometry fields. multi_mode="panel"
# tells HEDMGeometry these are FF multi-detector panels, not NF.
geom = HEDMGeometry(
    Lsd        = [d.lsd  for d in detectors_p1],
    y_BC       = [d.y_bc for d in detectors_p1],
    z_BC       = [d.z_bc for d in detectors_p1],
    tx         = [d.tx   for d in detectors_p1],   # survey guesses
    ty         = [d.ty   for d in detectors_p1],
    tz         = [d.tz   for d in detectors_p1],
    px         = INITIAL['px'],
    omega_start= OMEGA_START,
    omega_step = OMEGA_STEP,
    n_frames   = 1440,
    n_pixels_y = INITIAL['NrPixelsY'],
    n_pixels_z = INITIAL['NrPixelsZ'],
    wavelength = INITIAL['Wavelength'],
    flip_y     = True,
    apply_tilts= True,
    multi_mode = 'panel',
    wedge      = 0.0,                 # initial guess; refine below if desired
)

# Build the HKL table for the SAMPLE material (the indexed grains in pass 1
# are sample grains, not calibrant); cart/thetas/int_hkls come from a small
# helper analogous to build_hkls() in run_4det_refinement.py.
cart, thetas, int_hkls = build_hkls(  # noqa: F821 — supply your own (see paper script)
    SAMPLE['LatticeConstant'][0], INITIAL['Wavelength'], n_rings_used=7
)
model = HEDMForwardModel(cart, thetas, geom, hkls_int=int_hkls)

# ── (2) Pass-1 grains as autograd leaves ──────────────────────────────────
eulers    = torch.tensor(grains_p1[['EulerX','EulerY','EulerZ']].to_numpy(np.float32),
                         requires_grad=True)
positions = torch.tensor(grains_p1[['X','Y','Z']].to_numpy(np.float32),
                         requires_grad=True)
strains   = torch.zeros((len(grains_p1), 6), dtype=torch.float32, requires_grad=True)
latc      = torch.tensor([SAMPLE['LatticeConstant']]*len(grains_p1), dtype=torch.float32)

# ── (3) Which geometry params to refine ───────────────────────────────────
# free_setup is the same vocabulary as configure_freedom() in the paper script.
free_setup = ['tx_ty_tz']            # add 'Lsd', 'BC', 'wedge' as appropriate
refine_wedge = True                  # set True to also pick up global Wedge
tx_only      = True                  # if True, mask ty/tz gradient on model.tilts

model.tilts.requires_grad_(True)
model._Lsd_delta_mm.requires_grad_('Lsd' in free_setup)
model._y_BC.requires_grad_('BC' in free_setup)
model._z_BC.requires_grad_('BC' in free_setup)
model.wedge.requires_grad_(refine_wedge)

if tx_only:
    # Zero the ty/tz columns of model.tilts.grad on every backward pass.
    def _mask_ty_tz(grad):
        out = grad.clone()
        out[:, 1:] = 0.0
        return out
    model.tilts.register_hook(_mask_ty_tz)

# ── (4) Observed-spot bookkeeping ─────────────────────────────────────────
# SpotIndex (paper script line 222) maps pass-1 spots → (grain_idx, k_idx, m_idx)
# so the loss is an elementwise difference against the forward-model output.
# In production, midas_fit_grain.spec_residual.hedm_spot_residual replaces this.
from midas_fit_grain.spec_residual import hedm_spot_residual, HEDMResidualBundle
bundle = HEDMResidualBundle(  # see midas_fit_grain/spec_residual.py for the canonical builder
    model=model,
    observations=pipe_p1.layer_dir,   # per-panel InputAll.csv / Spots.bin live here
    matches=grains_p1,                # pass-1 grain assignments
    kind='pixel',
)

def loss_fn():
    unpacked = {
        'tilts_per_det': model.tilts,                # autograd link
        'Lsd_per_det'  : model._Lsd_delta_mm,
        'BC_y_per_det' : model._y_BC,
        'BC_z_per_det' : model._z_BC,
        'Wedge'        : model.wedge,
        'grain_euler'  : eulers,
        'grain_pos'    : positions,
        'grain_lattice': latc,
        'grain_strain' : strains,
    }
    return hedm_spot_residual(unpacked, bundle).pow(2).mean()

# ── (5) Adam warmup → L-BFGS polish ───────────────────────────────────────
setup_params = [p for p in
                (model.tilts, model._Lsd_delta_mm, model._y_BC, model._z_BC, model.wedge)
                if p.requires_grad]

opt = torch.optim.Adam(
    [{'params': setup_params,                'lr': 1e-3},   # tilts grad ~50× larger
     {'params': [eulers, positions, strains], 'lr': 5e-2}],
)
for it in range(200):
    opt.zero_grad(); L = loss_fn(); L.backward(); opt.step()
    if it % 50 == 0: print(f'adam  it={it:4d}  loss={L.item():.4e}')

opt = torch.optim.LBFGS(setup_params + [eulers, positions, strains],
                        lr=0.5, max_iter=50, history_size=20,
                        tolerance_grad=1e-9, tolerance_change=1e-12,
                        line_search_fn='strong_wolfe')
def _closure():
    opt.zero_grad(); L = loss_fn(); L.backward(); return L
for stage in range(4):
    L = opt.step(_closure)
    print(f'lbfgs stage={stage}  loss={float(L):.4e}')

# ── (6) Read out refined geometry ─────────────────────────────────────────
tx_refined    = model.tilts[:, 0].detach().cpu().numpy()        # degrees, per panel
ty_refined    = model.tilts[:, 1].detach().cpu().numpy()
tz_refined    = model.tilts[:, 2].detach().cpu().numpy()
wedge_refined = float(model.wedge.detach().cpu())

for i, d in enumerate(detectors_p1):
    print(f'  det {d.det_id}: tx {d.tx:+7.3f}° → {tx_refined[i]:+7.3f}°   '
          f'(Δ={tx_refined[i]-d.tx:+6.3f}°)')
print(f'Wedge: {wedge_refined:+.4f}°')

### Write the refined `detectors.json` for pass 2

In [ ]:
detectors_p2 = []
for i, d in enumerate(detectors_p1):
    # _Lsd_delta_mm is a per-panel mm-scale offset on top of the original Lsd;
    # physical Lsd_refined = Lsd + 1000 * _Lsd_delta_mm  (μm + mm→μm).
    lsd_delta_um = float(model._Lsd_delta_mm[i].detach().cpu()) * 1000.0
    detectors_p2.append(DetectorConfig(
        det_id   = d.det_id,
        zarr_path= d.zarr_path,
        lsd      = d.lsd  + lsd_delta_um if model._Lsd_delta_mm.requires_grad else d.lsd,
        y_bc     = float(model._y_BC[i].detach().cpu()) if model._y_BC.requires_grad else d.y_bc,
        z_bc     = float(model._z_BC[i].detach().cpu()) if model._z_BC.requires_grad else d.z_bc,
        tx       = float(tx_refined[i]),
        ty       = float(ty_refined[i]),
        tz       = float(tz_refined[i]),
        p_distortion = d.p_distortion,   # powder-derived; not refined here
    ))

detectors_json_p2 = WORK / 'detectors_pass2.json'
DetectorConfig.dump_many(detectors_p2, detectors_json_p2)
print(f'wrote {detectors_json_p2}  (Wedge={wedge_refined:+.4f}°)')
for d_old, d_new in zip(detectors_p1, detectors_p2):
    dtx = d_new.tx - d_old.tx
    print(f'  det {d_new.det_id}: tx {d_old.tx:+7.3f}° → {d_new.tx:+7.3f}°  (Δ={dtx:+6.3f}°)')

## 10 — `Parameters.txt` for **pass 2** (TIGHT tolerances)

Now that geometry is good, we can tighten the spot-match margins back to production values. Looser values cost us purity (false matches) and runtime; tightening them recovers both.

In [ ]:
params_pass2 = WORK / 'Parameters_pass2.txt'

tight_extra = f'''
Material         {SAMPLE['Material']}
LatticeConstant  {' '.join(map(str, SAMPLE['LatticeConstant']))}
SpaceGroup       {SAMPLE['SpaceGroup']}

OmegaStart      {OMEGA_START}
OmegaStep       {OMEGA_STEP}
OmegaRange      {OMEGA_START}  {OMEGA_START + (1440 * OMEGA_STEP)}
BoxSize         -1000000 1000000 -1000000 1000000
Wedge           {wedge_refined}

# --- TIGHT tolerances for pass-2 (production) -----------------------------
MinNrSpots       60
MarginOme        0.5
MarginEta        500
MarginRadial     400
MarginRadius     500
StepSizePos      100
StepSizeOrient   0.2
DiscModel        0
tolTilts         2.0
tolBC           10.0
tolLsd         500.0
RingsToUse       1 2 3 4 5 6 7 8
'''
params_pass2.write_text(params_template.read_text() + tight_extra)
print(f'wrote {params_pass2}')

## 11 — Run the pipeline (pass 2)

In [ ]:
pipe_p2 = Pipeline(
    config=PipelineConfig(
        result_dir     = str(WORK / 'run_pass2'),
        params_file    = str(params_pass2),
        detectors_json = str(detectors_json_p2),
        n_cpus         = 8,
        device         = 'cpu',
        dtype          = 'float64',
    ),
    detectors=detectors_p2,
)
pipe_p2.run()
result_p2 = pipe_p2.layer_result
grains_p2 = result_p2.grains_df()

## 12 — Compare passes

In [ ]:
print(f'                pass 1 (loose)   pass 2 (tight)')
print(f'grains          {len(grains_p1):>14d} {len(grains_p2):>15d}')
print(f'total spots     {result_p1.cross_det_merge.n_total_spots:>14,d} {result_p2.cross_det_merge.n_total_spots:>15,d}')
print(f'runtime (s)     {result_p1.total_duration_s():>14.1f} {result_p2.total_duration_s():>15.1f}')

if 'NrSpotsMatched' in grains_p1.columns:
    print(f'mean spots/grain {grains_p1["NrSpotsMatched"].mean():>13.1f} {grains_p2["NrSpotsMatched"].mean():>15.1f}')
if 'meanRadius' in grains_p1.columns:
    print(f'median strain    {grains_p1["meanInternalAngle"].median():>13.4f} {grains_p2["meanInternalAngle"].median():>15.4f}')

# Per-panel spot counts side-by-side
print('\nspots per panel:')
for (d1, n1), (d2, n2) in zip(zip(detectors_p1, result_p1.cross_det_merge.n_per_detector),
                              zip(detectors_p2, result_p2.cross_det_merge.n_per_detector)):
    print(f'  det {d1.det_id}: {n1:>8,d} (tx {d1.tx:+6.2f}°)  →  {n2:>8,d} (tx {d2.tx:+6.2f}°)')

## What this notebook does **not** cover

- **More than two bootstrap iterations.** If pass-2 still shows asymmetric per-panel spot counts, run another (HEDM-only) refine → pipeline cycle with the pass-2 grains. Convergence is usually fast (2–3 iterations).
- **Strain reliability.** The L-BFGS step deliberately leaves grain strains as autograd leaves that absorb residual energy alongside `tx/Wedge`. Strain values read off this stage are **not trustworthy** — re-run the pipeline (pass 2) and read strain from the `process_grains` output, where geometry is fixed.
- **Refining powder-set parameters from HEDM.** `Lsd`, `y_BC`, `z_BC`, `ty`, `tz`, distortion are *better* constrained by powder than by a few-grain HEDM dataset — only thaw `Lsd`/`BC` in step 9 if you have a grain-rich dataset and have a specific reason to suspect drift between calibration and HEDM acquisition.
- **Non-TIFF formats.** `ffGenerateZipRefactor.py` auto-dispatches on extension (TIFF / GE5 / HDF5 / CBF). For unusual formats or ad-hoc inspection, **fabIO** (`pip install fabio`) opens essentially every detector format with a uniform `.data` / `.header` API.
- **Joint powder + HEDM refinement.** A separate package (`midas_joint_ff_calibrate`) explores running both residuals simultaneously, but that's a research direction, not the production workflow used here. This notebook keeps powder and HEDM strictly sequential.

## Where to go next

Once pass-2 has converged, the `grains_p2` table + `detectors_pass2.json` + pipeline outputs are the standard inputs to the rest of the multi-detector analysis tooling. The companion notebook [`03_multi_detector_demo.ipynb`](03_multi_detector_demo.ipynb) walks through the downstream analysis on a converged 4-panel pipeline result — plotting per-grain stats, mining the `cross_det_merge` outputs, comparing grain populations across panels, and the standard QC plots. Pick up there with `result = pipe_p2.layer_result` (or `grains_p2 = result_p2.grains_df()`) and follow that notebook's cells.